# Tenent Settings Best Practics Analyzer

When you run this notebook, the Tenant Settings Best Practice Analyzer (TSBPA) will offer tips to improve your tenant settings.

The TSBPA checks against recommendations for the current (2026-06-01) 167 tenant settings. These recommendations come from experts within the Fabric Community.

You’ll get suggestions for improvement in the following categories: 
* Additional workloads
* Admin API settings
* Advanced networking
* App settings
* Audit and usage settings
* Azure AI Service
* Azure Maps services
* Copilot and Azure OpenAI Service
* Dashboard settings
* Datamart settings
* Developer settings
* Discovery settings
* Domain management settings
* Encryption
* Explore settings (preview)
* Export and sharing settings
* Gen1 dataflow settings
* Git integration
* Help and support settings
* Information protection
* Insights settings
* Integration settings
* Microsoft Fabric
* OneLake settings
* Power BI visuals
* Q&A settings
* R and Python visuals settings
* Scale-out settings
* Scorecards settings
* Semantic Model Security
* Semantic model settings
* Share data with your Microsoft 365 services
* Template app settings
* User experience experiments
* Workspace settings

## Powering this feature: Semantic Link (Lab)
This notebook leverages [Semantic Link](https://learn.microsoft.com/fabric/data-science/semantic-link-overview) and [Semantic Link Lab Admin](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.admin.html), python libraries which lets you query and update  Fabric items for different use cases. The "[list_tenant_settings](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.admin.html#sempy_labs.admin.list_tenant_settings)" and "[update_tenant_setting](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.admin.html#sempy_labs.admin.update_tenant_setting)" functions used in this notebook are just one example of the useful [functions]((https://learn.microsoft.com/python/api/semantic-link-sempy/sempy.fabric)) which Semantic Link and Semantic Link Labs offers.

You can find more [functions](https://github.com/microsoft/semantic-link-labs#featured-scenarios) and [helper notebooks](https://github.com/microsoft/semantic-link-labs/tree/main/notebooks) in [Semantic Link Labs](https://github.com/microsoft/semantic-link-labs), a Python library that extends Semantic Link's capabilities to automate technical tasks.

## Low-code solutions for data tasks
You don't have to be a Python expert to use Semantic Link or Semantic Link Labs. Many functions can be used simply by entering your parameters and running the notebook.


## Install and import libraries

In [33]:
# Install SemPy
%pip install semantic-link
%pip install semantic-link-labs


StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 55, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [34]:
# import the module
from sempy_labs import admin 
import pandas as pd

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 57, Finished, Available, Finished, False)

## Parameter

In [35]:
recommendation_mode = "Light" # "Light" or "Paranoid"
recommendation_path_and_file = "/lakehouse/default/Files/tenant_settings_recommendations.csv"
recommendation_file_delimiter = ";"
debug_mode = False

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 58, Finished, Available, Finished, False)

## Defaults for missing parameters

In [36]:
#recommendation_mode = globals().get("recommendation_mode", "Light")
#recommendation_path_and_file = globals().get("recommendation_path_and_file", "/lakehouse/default/Files/tenant_settings_recommendations.csv")
#recommendation_file_delimiter = globals().get("recommendation_file_delimiter", ";")
#debug_mode = globals().get("debug_mode", "False")
 
print(f"recommendation_mode: '{recommendation_mode}'")
print(f"recommendation_path_and_file: '{recommendation_path_and_file}'")
print(f"recommendation_file_delimiter: '{recommendation_file_delimiter}'")
print(f"debug_mode: '{debug_mode}'")

# Derive variables from parameter
recommendation = f"Best Practice Recommendation {recommendation_mode}"
print(f"recommendation: '{recommendation}'")

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 59, Finished, Available, Finished, False)

recommendation_mode: 'Light'
recommendation_path_and_file: '/lakehouse/default/Files/tenant_settings_recommendations.csv'
recommendation_file_delimiter: ';'
debug_mode: 'False'
recommendation: 'Best Practice Recommendation Light'


## Ingest general recommendations from CSV

In [37]:
tenant_settings_recommendation = pd.read_csv(
    recommendation_path_and_file, 
    delimiter = recommendation_file_delimiter)
if debug_mode==True:
    display(tenant_settings_recommendation)


StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 60, Finished, Available, Finished, False)

## Define general function 

In [38]:
def get_recommended_settings_script(
    level,
    tenant_settings,
    tenant_settings_recommendation,
    recommendation,
    check_missing_settings,
    function_name,
    setting_name,
    title_settings_name,
    enabled_settings_name,
    debug_mode=False
):
    """Generate SemPy admin.* update commands for mismatched settings.

    Key robustness improvements:
    - Safely handles cases where the merged DataFrame does NOT contain
      a `Title_recommendation` column (e.g. capacity-level overrides).
    - Only checks for "missing recommendations" when the column exists
      and when `check_missing_settings` is True.
    - Safely handles cases where the expected "enabled" column name
      (enabled_settings_name) does not exist in the merged DataFrame.
    """

    if tenant_settings.empty:
        print(f"No settings for {level}, no recommendations.")
        return
    else:
        print(f"# Recommended changes for your {level} settings:")

    # Merge settings with recommendation metadata
    tenant_settings = tenant_settings.merge(
        tenant_settings_recommendation,
        on=setting_name,
        how="left",
        suffixes=("_settings", "_recommendation"),
    )

    if debug_mode:
        print("# Columns after merge:")
        print(list(tenant_settings.columns))
        display(tenant_settings)

    # ---------------------------------------------------------------
    # Handle missing recommendations safely
    # Some levels (e.g. Capacity) may not produce a Title_recommendation
    # column after the merge. Guard the access to avoid KeyError.
    # ---------------------------------------------------------------
    if check_missing_settings and "Title_recommendation" in tenant_settings.columns:
        tenant_settings_missing = tenant_settings.loc[
            tenant_settings["Title_recommendation"].isna()
            | tenant_settings["Title_recommendation"].eq(""),
            [setting_name, title_settings_name, enabled_settings_name],
        ]

        if not tenant_settings_missing.empty:
            print("##################################################################")
            print("# The recommendation file does not contain the following settings:")
            print(f"# {tenant_settings_missing}")
            print("# Take a close look at these settings manually.")
            print("##################################################################")
    elif check_missing_settings and "Title_recommendation" not in tenant_settings.columns:
        # Only emit a light warning; this is expected for some override tables
        print("# Note: No 'Title_recommendation' column found after merge; "
              "skipping \"missing recommendations\" check for this level.")

    # ---------------------------------------------------------------
    # Ensure the expected "enabled" column exists before proceeding
    # ---------------------------------------------------------------
    if enabled_settings_name not in tenant_settings.columns:
        print(
            f"# WARNING: Expected enabled column '{enabled_settings_name}' not found "
            f"for level '{level}'. Available columns: {list(tenant_settings.columns)}"
        )
        print("# Skipping automatic recommendation generation for this level.")
        return

    # Print install instructions (for generated script reuse)
    print("# Install SemPy")
    print("%pip install semantic-link-labs\n")
    print("# import the module")
    print("from sempy_labs import admin\n")

    # Filter settings where a recommendation exists and differs from current
    col = tenant_settings[f"{recommendation}"]
    tenant_settings_filtered = tenant_settings.loc[
        col.notna()
        & col.astype(str).ne("")
        & col.astype(str).str.lower().ne("n/a")
        & tenant_settings[enabled_settings_name].ne(col),
        [setting_name, title_settings_name, enabled_settings_name, recommendation],
    ]

    # Generate update commands
    for _, tenant_setting in tenant_settings_filtered.iterrows():
        if level != "" and level != "Tenant":
            id_column_name = f"{level} Id"
            id_column_text = f'"{tenant_setting[id_column_name]}", '
        else:
            id_column_name = ""
            id_column_text = ""

        print(
            f'admin.{function_name}('
            f"{id_column_text}"
            f'"{tenant_setting[setting_name]}", '
            f'"{tenant_setting[recommendation]}") '
            f"# {tenant_setting[title_settings_name]}"
        )

    # No explicit return needed; function prints script lines only

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 61, Finished, Available, Finished, False)

## Generate recommendations

In [39]:
debug_mode = True

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 62, Finished, Available, Finished, False)

### Tenant Level

In [40]:
tenant_settings = admin.list_tenant_settings()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Tenant",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_tenant_setting",
    setting_name="Setting Name",
    title_settings_name="Title_settings",
    enabled_settings_name="Enabled_settings",
    debug_mode=debug_mode
)

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 315c9acf-0103-44b4-814e-bc7d18eab1e1)

# Recommended changes for your Tenant settings:
# Columns after merge:
['Setting Name', 'Title_settings', 'Enabled_settings', 'Can Specify Security Groups_settings', 'Tenant Setting Group_settings', 'Enabled Security Groups_settings', 'Excluded Security Groups_settings', 'Delegate To Capacity_settings', 'Delegate To Workspace_settings', 'Delegate To Domain_settings', 'Properties_settings', 'Title_recommendation', 'Enabled_recommendation', 'Can Specify Security Groups_recommendation', 'Tenant Setting Group_recommendation', 'Enabled Security Groups_recommendation', 'Excluded Security Groups_recommendation', 'Delegate To Capacity_recommendation', 'Delegate To Workspace_recommendation', 'Delegate To Domain_recommendation', 'Properties_recommendation', 'Best Practice Recommendation Light', 'Best Practice Recommendation Light Default', 'Best Practice Recommendation Paranoid', 'Best Practice Recommendation Paranoid Default']


SynapseWidget(Synapse.DataFrame, f9858d1b-cb63-41bb-b5c4-f5db0d44ee9d)

# Install SemPy
%pip install semantic-link-labs

# import the module
from sempy_labs import admin

admin.update_tenant_setting("CertifiedCustomVisualsTenant", "True") # Add and use certified visuals only (block uncertified)
admin.update_tenant_setting("DatamartTenant", "False") # Create Datamarts (preview)
admin.update_tenant_setting("PublishToWeb", "False") # Publish to web


### Capacity level

In [41]:
tenant_settings = admin.list_capacity_tenant_settings_overrides()
if debug_mode==True:
    display(tenant_settings)

# For capacity-level settings, the merged DataFrame may not contain a
# 'Title_recommendation' column (only 'Title'), which causes a KeyError
# in get_recommended_settings_script when it tries to check for
# missing recommendations. We therefore disable the "missing settings"
# check for this level.
get_recommended_settings_script(
    level="Capacity",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=False,  # avoid KeyError on non-existent Title_recommendation column
    function_name="update_capacity_tenant_setting_override",
    setting_name="Setting Name",
    title_settings_name="Setting Title_settings",
    enabled_settings_name="Setting Enabled_settings",
    debug_mode=True
)

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 408ff90f-e374-4c35-96a0-b45d07f21683)

# Recommended changes for your Capacity settings:
# Columns after merge:
['Capacity Id', 'Setting Name', 'Setting Title', 'Setting Enabled', 'Can Specify Security Groups_settings', 'Enabled Security Groups_settings', 'Excluded Security Groups_settings', 'Tenant Setting Group_settings', 'Tenant Setting Properties', 'Delegate to Workspace', 'Delegated From', 'Title', 'Enabled', 'Can Specify Security Groups_recommendation', 'Tenant Setting Group_recommendation', 'Enabled Security Groups_recommendation', 'Excluded Security Groups_recommendation', 'Delegate To Capacity', 'Delegate To Workspace', 'Delegate To Domain', 'Properties', 'Best Practice Recommendation Light', 'Best Practice Recommendation Light Default', 'Best Practice Recommendation Paranoid', 'Best Practice Recommendation Paranoid Default']


SynapseWidget(Synapse.DataFrame, bf4f865f-9339-482b-9339-27f407ad4015)

# WARNING: Expected enabled column 'Setting Enabled_settings' not found for level 'Capacity'. Available columns: ['Capacity Id', 'Setting Name', 'Setting Title', 'Setting Enabled', 'Can Specify Security Groups_settings', 'Enabled Security Groups_settings', 'Excluded Security Groups_settings', 'Tenant Setting Group_settings', 'Tenant Setting Properties', 'Delegate to Workspace', 'Delegated From', 'Title', 'Enabled', 'Can Specify Security Groups_recommendation', 'Tenant Setting Group_recommendation', 'Enabled Security Groups_recommendation', 'Excluded Security Groups_recommendation', 'Delegate To Capacity', 'Delegate To Workspace', 'Delegate To Domain', 'Properties', 'Best Practice Recommendation Light', 'Best Practice Recommendation Light Default', 'Best Practice Recommendation Paranoid', 'Best Practice Recommendation Paranoid Default']
# Skipping automatic recommendation generation for this level.


### Domain level

In [42]:
tenant_settings = admin.list_domain_tenant_settings_overrides()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Domain",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_domain_tenant_setting_override",
    setting_name="Setting Name",
    title_settings_name="Title_settings",
    enabled_settings_name="Enabled_settings",
    debug_mode=debug_mode
)

StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 65, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dcd0d3ee-6751-4229-9984-aa3faf35f99e)

# Recommended changes for your Domain settings:
# Columns after merge:
['Domain Id', 'Setting Name', 'Title_settings', 'Enabled_settings', 'Can Specify Security Groups_settings', 'Enabled Security Groups_settings', 'Tenant Setting Group_settings', 'Delegated To Workspace', 'Delegated From', 'Title_recommendation', 'Enabled_recommendation', 'Can Specify Security Groups_recommendation', 'Tenant Setting Group_recommendation', 'Enabled Security Groups_recommendation', 'Excluded Security Groups', 'Delegate To Capacity', 'Delegate To Workspace', 'Delegate To Domain', 'Properties', 'Best Practice Recommendation Light', 'Best Practice Recommendation Light Default', 'Best Practice Recommendation Paranoid', 'Best Practice Recommendation Paranoid Default']


SynapseWidget(Synapse.DataFrame, 76143fca-dfb8-4539-a7d8-35748c79eba7)

# Install SemPy
%pip install semantic-link-labs

# import the module
from sempy_labs import admin



### Workspace level

In [43]:
tenant_settings = admin.list_workspaces_tenant_settings_overrides()
if debug_mode==True:
    display(tenant_settings)
get_recommended_settings_script(
    level="Workspace",
    tenant_settings=tenant_settings, 
    tenant_settings_recommendation=tenant_settings_recommendation, 
    recommendation=recommendation, 
    check_missing_settings=True,
    function_name="update_workspace_tenant_setting_override",
    setting_name="Setting Name",
    title_settings_name="Title_settings",
    enabled_settings_name="Enabled_settings",
    debug_mode=debug_mode
)


StatementMeta(, ae2f7357-4d1b-4a66-90f7-7a2b09d4f1e7, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 33c4a9a7-2fa1-41ad-bfc1-c81a648675f7)

# Recommended changes for your Workspace settings:
# Columns after merge:
['Workspace Id', 'Setting Name', 'Title_settings', 'Enabled_settings', 'Can Specify Security Groups_settings', 'Enabled Security Groups_settings', 'Tenant Setting Group_settings', 'Delegated From', 'Title_recommendation', 'Enabled_recommendation', 'Can Specify Security Groups_recommendation', 'Tenant Setting Group_recommendation', 'Enabled Security Groups_recommendation', 'Excluded Security Groups', 'Delegate To Capacity', 'Delegate To Workspace', 'Delegate To Domain', 'Properties', 'Best Practice Recommendation Light', 'Best Practice Recommendation Light Default', 'Best Practice Recommendation Paranoid', 'Best Practice Recommendation Paranoid Default']


SynapseWidget(Synapse.DataFrame, 3711f856-c698-450b-b589-8885cd4e8e46)

# Install SemPy
%pip install semantic-link-labs

# import the module
from sempy_labs import admin

